# CINEMA Asset Generator


In [46]:
import pandas as pd
import os

# --- CONFIGURATION ---
INPUT_CSV = '2hr/2hr/cartesian_elems_umbra_flag_CA_20260306115024.txt'  # Your original file name
OUTPUT_ASSET = 'CINEMAtest.asset'
GLB_FILENAME = 'sat.glb'               # Must match your .glb file name
IDENTIFIER = 'MySatellite'             # Unique ID for the node

def generate_asset():
    # Load data
    df = pd.read_csv(INPUT_CSV)

    # Downsample df; no need to overload the rendering pipeline.
    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    df = df.dropna(subset=['Time']) # Drop rows where Time might be null to avoid Index errors
    df.set_index('Time', inplace=True)
    df = df.resample('2 min').mean()
    
    df= df.reset_index()
    # Ensure Time is in the right format: 2030-07-04T00:00:00.00Z
    df['Time'] = pd.to_datetime(df['Time']).dt.strftime('%Y-%m-%dT%H:%M:%S.00Z')

    # Reduce number of keyframes to something more manageable
    df = df.head(5000)
    
    # Get Start and End times for the TimeFrameInterval
    start_time = df['Time'].iloc[0]
    end_time = df['Time'].iloc[-1]

    # Start building the Lua string
    lua = []
    lua.append('local sun = asset.require("scene/solarsystem/sun/transforms")')
    lua.append('local earthTransforms = asset.require("scene/solarsystem/planets/earth/earth")\n')
    
    # 1. Keyframes Table
    lua.append('-- 1. Keyframes')
    lua.append('local myKeyframes = {')
    for _, row in df.iterrows():
        # Scale km to meters
        x, y, z = row['x_J2000']*1000, row['y_J2000']*1000, row['z_J2000']*1000
        line = f'    ["{row["Time"]}"] = {{ Type = "StaticTranslation", Position = {{ {x:.2f}, {y:.2f}, {z:.2f} }} }},'
        lua.append(line)
    lua.append('}\n')

    # 2. Position Node
    lua.append('-- 2. Position Node')
    lua.append(f'local satellitePosition = {{')
    lua.append(f'    Identifier = "{IDENTIFIER}_Pos",')
    lua.append(f'    Parent = "Earth",')
    lua.append(f'    TimeFrame = {{ Type = "TimeFrameInterval", Start = "{start_time}", End = "{end_time}" }},')
    lua.append(f'    Transform = {{ Translation = {{ Type = "TimelineTranslation", Keyframes = myKeyframes }} }},')
    lua.append(f'    GUI = {{ Name = "{IDENTIFIER} Position", Path = "/CINEMA/Positions" }}')
    lua.append(f'}}\n')

    # 3. Model Node
    lua.append('-- 3. Renderable Model')
    lua.append(f'local satelliteModel = {{')
    lua.append(f'    Identifier = "{IDENTIFIER}_Model",')
    lua.append(f'    Parent = "{IDENTIFIER}_Pos",')
    lua.append(f'    Transform = {{ Scale = {{ Type = "StaticScale", Scale = 1000.0 }} }},')
    lua.append(f'    Renderable = {{')
    lua.append(f'        Type = "RenderableModel",')
    lua.append(f'        GeometryFile = asset.resource("./{GLB_FILENAME}"),')
    lua.append(f'        BoundingSphere = 100.0,')
    lua.append(f'        LightSources = {{')
    lua.append(f'            sun.LightSource,')
    lua.append(f'            {{ Identifier = "{IDENTIFIER}_CamLight", Type = "CameraLightSource", Intensity = 0.5 }}')
    lua.append(f'        }}')
    lua.append(f'    }},')
    lua.append(f'    GUI = {{ Name = "{IDENTIFIER} Model", Path = "/CINEMA/Models" }}')
    lua.append(f'}}\n')

    # 4. Trail Node
    lua.append('-- 4. Trail')
    lua.append(f'local satelliteTrail = {{')
    lua.append(f'    Identifier = "{IDENTIFIER}_Trail",')
    lua.append(f'    Parent = "Earth",')
    lua.append(f'    Renderable = {{')
    lua.append(f'        Type = "RenderableTrailTrajectory",')
    lua.append(f'        Enabled = true,')
    lua.append(f'        Translation = {{ Type = "TimelineTranslation", Keyframes = myKeyframes }},')
    lua.append(f'        Color = { {color_str} },')
    lua.append(f'        StartTime = "{start_time}",')
    lua.append(f'        EndTime = "{end_time}",')
    lua.append(f'        SampleInterval = 10')
    lua.append(f'    }},')
    lua.append(f'    GUI = {{ Name = "{IDENTIFIER} Trail", Path = "/CINEMA/Trails" }}')
    lua.append(f'}}\n')

    # 5. Initialization logic
    lua.append('asset.onInitialize(function()')
    lua.append(f'    openspace.addSceneGraphNode(satellitePosition)')
    lua.append(f'    openspace.addSceneGraphNode(satelliteModel)')
    lua.append(f'    openspace.addSceneGraphNode(satelliteTrail)')
    lua.append('end)\n')
    
    lua.append('asset.onDeinitialize(function()')
    lua.append(f'    openspace.removeSceneGraphNode(satelliteTrail)')
    lua.append(f'    openspace.removeSceneGraphNode(satelliteModel)')
    lua.append(f'    openspace.removeSceneGraphNode(satellitePosition)')
    lua.append('end)')

    # Write to file
    with open(OUTPUT_ASSET, 'w') as f:
        f.write("\n".join(lua))
    print(f"Successfully created {OUTPUT_ASSET}")

if __name__ == "__main__":
    generate_asset()

Successfully created CINEMAtest.asset


In [ ]:
# --- CONFIGURATION ---

ID_list = ["CA", "CF", "CM", "EA", "EF", "EM", "WA", "WF", "WM"]
import matplotlib.pyplot as plt
cmap = plt.get_cmap('hsv').resampled(10)


# for i in range(10):
#     color = cmap(i) # Access color directly by index
#     color_str = str(color[0]) + "," + str(color[1]) + "," + str(color[2])
#     print(color_str)
    

for i in range(10):
    ID = ID_list[i]
    color = cmap(i)
    color_str = str(color[0]) + "," + str(color[1]) + "," + str(color[2])
    IDENTIFIER = ID
    INPUT_CSV = '2hr/2hr/cartesian_elems_umbra_flag_' + ID + '_20260306115024.txt'  # Your original file name
    OUTPUT_ASSET = 'CINEMA_' + ID + '.asset'
    GLB_FILENAME = 'sat.glb'               # Must match your .glb file name
    generate_asset()



In [26]:
# from harkness.create_ply import create_ply
# df = pd.read_csv(INPUT_CSV)

# # Downsample df; no need to overload the rendering pipeline.
# df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
# df = df.dropna(subset=['Time']) # Drop rows where Time might be null to avoid Index errors
# df.set_index('Time', inplace=True)
# df = df.resample('120T').mean()

# df= df.reset_index()
# # Ensure Time is in the right format: 2030-07-04T00:00:00.00Z
# df['Time'] = pd.to_datetime(df['Time']).dt.strftime('%Y-%m-%dT%H:%M:%S.00Z')
# # Create PLY file
# filename = 'example.ply'
# # create_ply(df, filename, C='C', is_verbose=True)
# create_ply(df, filename, C='Time', X='x_J2000', Y='y_J2000', Z='z_J2000', cmap='viridis', is_verbose=False)
# # px.scatter_3d(df.x_J2000, df.y_J2000, df.z_J2000)

/tmp/ipykernel_1843/743238439.py:8: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.

